In [ ]:
#@title 01  ·  llama.cpp { display-mode: "form" }
build_url = "https://github.com/GodL-x-SouL/SenseNova-Colab/releases/latest/download/SenseNova-llama-server-cuda-sm75.zip" #@param {type:"string"}
force_download = False #@param {type:"boolean"}

import os, re, sys, json, time, html, shutil, zipfile, subprocess, warnings
import urllib.request
warnings.filterwarnings("ignore")

from IPython.display import display, HTML

ROOT = "/content/SenseNovaLoop"
REPO = os.path.join(ROOT, "llama.cpp-loop")
BIN_DIR = os.path.join(REPO, "build", "bin")
BIN = os.path.join(BIN_DIR, "llama-server")
MARK = os.path.join(ROOT, "build.json")
ZIP = os.path.join(ROOT, "SenseNova-llama-server-cuda-sm75.zip")
COMMIT = "a7f77a788893ef2cb17c82b9ac0cb1279bcd0918"
os.makedirs(ROOT, exist_ok=True)

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-row{display:flex;gap:18px;justify-content:space-between;padding:5px 0;font-size:16px;line-height:1.4;}
.om-k{color:#8a8070;min-width:160px;} .om-v{color:#e8dfd0;text-align:right;}
.om-bar-track{height:2px;background:#2a261f;margin:14px 0 10px;} .om-bar-fill{height:2px;background:#b8956c;width:0;}
.om-log{background:#0e0c0a;border:1px solid #2a261f;padding:12px 14px;height:170px;max-height:170px;overflow:hidden;
font-family:"Courier New",Courier,monospace;font-size:11px;line-height:1.55;color:#c8bba6;white-space:pre-wrap;word-break:break-word;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

class OMBox:
    def __init__(self, no, title):
        self.no, self.title = no, title
        self.status, self.rows, self.foot, self.extra, self.pct = "WORKING", [], "", "", None
        self._h = None
        self.paint()
    def paint(self):
        rows = "".join(
            f'<div class="om-row"><span class="om-k">{esc(k)}</span><span class="om-v">{esc(v)}</span></div>'
            for k, v in self.rows
        )
        bar = f'<div class="om-bar-track"><div class="om-bar-fill" style="width:{self.pct:.1f}%"></div></div>' if self.pct is not None else ""
        h = HTML(CSS + f"""
        <div class="om-wrap">
          <div class="om-top"><div class="om-kicker">{esc(self.no)}</div>
            <div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div>
          <div class="om-title">{esc(self.title)}</div>
          <div class="om-rule"></div>
          {rows}{bar}{self.extra}
          <div class="om-foot">{esc(self.foot)}</div>
        </div>""")
        if self._h is None:
            self._h = display(h, display_id=f"sn-{self.no}")
        else:
            self._h.update(h)
    def set(self, **k):
        for a in ("rows", "status", "foot", "extra", "pct"):
            if a in k:
                setattr(self, a, k[a])
        self.paint()

def run(cmd, cwd=None, timeout=None):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout or "", p.stderr or ""

def logbox(text):
    return f'<div class="om-log">{esc(text)}</div>' if (text or "").strip() else ""

def tail(text, n=900):
    return (text or "").strip()[-n:]

def hsize(n):
    return f"{n / 1024**3:.2f} GB" if n >= 1024**3 else f"{n / 1024**2:.1f} MB"

ui = OMBox("01", "llama.cpp")
t0 = time.time()
rows = []

def elapsed():
    s = int(time.time() - t0)
    return f"{s//60}m {s%60:02d}s" if s >= 60 else f"{s}s"

def tick(key, val):
    for i, (k, _) in enumerate(rows):
        if k == key:
            rows[i] = (key, val)
            break
    else:
        rows.append((key, val))
    ui.set(rows=rows, foot=f"running  ·  {elapsed()}")

def fail(msg, detail=""):
    ui.set(status="ERROR", extra=logbox(detail), pct=None, foot=msg)
    raise SystemExit from None

tick("device", "detecting")
gpu, vram, cuda, arch = "cpu", "—", "—", ""
try:
    c, o, _ = run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits")
    if c == 0 and o.strip():
        gpu, vram = [x.strip() for x in o.strip().splitlines()[0].split(",")]
except Exception:
    pass
try:
    import torch
    if torch.cuda.is_available():
        arch = "%d%d" % torch.cuda.get_device_capability(0)
        cuda = str(torch.version.cuda or "—")
except Exception:
    pass
if not arch:
    c, o, _ = run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader,nounits")
    if c == 0 and o.strip():
        arch = o.strip().splitlines()[0].replace(".", "")
tick("device", f"{gpu}  ·  {vram} MB  ·  cuda {cuda}" + (f"  ·  sm{arch}" if arch else ""))

if not (os.path.isfile("/usr/local/bin/cloudflared") and os.access("/usr/local/bin/cloudflared", os.X_OK)):
    tick("tunnel", "cloudflared")
    run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    run("chmod +x /usr/local/bin/cloudflared")
if not os.access("/usr/local/bin/cloudflared", os.X_OK):
    fail("cloudflared install failed", "")
tick("tunnel", "cloudflared ready")

have = os.path.isfile(BIN) and os.path.isfile(MARK) and not force_download
if have:
    tick("release", "cached  ·  " + os.path.basename(ZIP))
else:
    tick("release", "downloading")
    part = ZIP + ".part"
    try:
        req = urllib.request.Request(build_url, headers={"User-Agent": "sensenova-colab"})
        with urllib.request.urlopen(req, timeout=60) as r, open(part, "wb") as f:
            total = int(r.headers.get("Content-Length") or 0)
            got, last_t, last_b = 0, time.time(), 0
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                got += len(chunk)
                if total:
                    ui.pct = got * 100.0 / total
                now = time.time()
                if now - last_t > 0.4:
                    speed = (got - last_b) / max(now - last_t, 1e-6) / 1e6
                    tick("release", f"{hsize(got)} / {hsize(total) if total else '?'}  ·  {speed:.1f} MB/s")
                    last_t, last_b = now, got
        os.replace(part, ZIP)
        ui.pct = None
    except Exception as e:
        if os.path.exists(part):
            os.remove(part)
        fail("release download failed", f"{build_url}\n{e}")
    tick("release", f"{os.path.basename(ZIP)}  ·  {hsize(os.path.getsize(ZIP))}")

    tick("build", "extracting")
    try:
        with zipfile.ZipFile(ZIP) as z:
            members = z.namelist()
            z.extractall(ROOT)
    except Exception as e:
        fail("unzip failed", str(e))
    if not os.path.isfile(BIN):
        fail("archive missing llama-server", f"expected  {BIN}\ngot: " + ", ".join(members[:8]))
    for dp, _, fs in os.walk(BIN_DIR):
        for f in fs:
            try:
                os.chmod(os.path.join(dp, f), 0o755)
            except Exception:
                pass
    with open(MARK, "w") as f:
        json.dump({"commit": COMMIT, "head": "", "cuda": True, "arch": arch or "75",
                   "build": "cuda sm75  ·  release", "at": time.strftime("%Y-%m-%d %H:%M")}, f)

nf, size = 0, 0
for dp, _, fs in os.walk(BIN_DIR):
    for f in fs:
        nf += 1
        size += os.path.getsize(os.path.join(dp, f))
tick("build", f"{nf} files  ·  {hsize(size)}")

tick("server", "verifying")
try:
    c, o, e = run([BIN, "--help"], timeout=60)
except Exception as ex:
    c, o, e = -1, "", str(ex)
help_all = (o or "") + (e or "")
missing = [f for f in ("--spec-type", "--prompt-tile", "--ctx-checkpoint-turns", "--spec-rollback") if f not in help_all]
if c != 0 or missing:
    detail = []
    if missing:
        detail.append("missing: " + "  ".join(missing))
    if e.strip():
        detail.append(tail(e))
    fail("server verification failed", "\n".join(detail))

tick("server", "ready  ·  flags verified")
ui.set(status="READY", extra="", pct=None, foot=f"imported in {elapsed()}  ·  next: cell 02")

In [ ]:
#@title 02  ·  Models { display-mode: "form" }
quant = "Q4_K_M" #@param ["Q4_K_M", "Q6_K"]
drafter = "drafter-2" #@param ["drafter-2", "drafter", "none"]
include_patches = True #@param {type:"boolean"}
skip_existing = True #@param {type:"boolean"}

import os, re, time, html, threading, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], capture_output=True)
    import requests
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import unquote, urlparse
from IPython.display import display, HTML

ROOT = "/content/SenseNovaLoop"
MODELS = os.path.join(ROOT, "models")
PATCHES = os.path.join(ROOT, "patches")
MARK = os.path.join(ROOT, "build.json")
HF_REPO = "satellitedown/SenseNova-Loop-9B-abliterated-GGUF"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve/main/"
DRAFTERS = {
    "drafter-2": "SenseNova-Loop-9B-abliterated-drafter-2-Q4_K_M.gguf",
    "drafter": "SenseNova-Loop-9B-abliterated-drafter-Q4_K_M.gguf",
    "none": "",
}

CSS = r"""
<style>
.om-wrap{width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
border:1px solid #3f382c;padding:28px 32px 22px;font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;}
.om-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.om-kicker{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}
.om-status{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}
.om-s-WORKING{color:#b8956c;} .om-s-READY,.om-s-LIVE{color:#8fa382;} .om-s-ERROR{color:#8a3a3a;} .om-s-STOPPED{color:#8a8070;}
.om-title{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}
.om-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.om-name{font-size:18px;color:#f3eadb;} .om-sub{color:#8a8070;font-size:14px;margin-top:2px;}
.om-bar-track{height:2px;background:#2a261f;margin:16px 0 10px;} .om-bar-fill{height:2px;background:#b8956c;}
.om-stat{font-family:"Courier New",Courier,monospace;font-size:11px;color:#a89880;letter-spacing:.04em;}
.om-hist{margin-top:16px;} .om-hrow{display:flex;justify-content:space-between;gap:16px;padding:3px 0;font-size:15px;}
.om-hk{color:#8a8070;} .om-hv{color:#c8bba6;}
.om-foot{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}
.om-note{color:#a89880;font-size:15px;line-height:1.5;}
</style>
"""

def esc(s):
    return html.escape(str(s) if s is not None else "")

def fmt_bytes(n):
    n = float(n or 0)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or u == "TB":
            return f"{n:.0f} {u}" if u == "B" else f"{n:.2f} {u}"
        n /= 1024.0

def fmt_eta(sec):
    if sec is None or sec == float("inf") or sec < 0:
        return "—"
    sec = int(sec)
    if sec < 60: return f"{sec}s"
    if sec < 3600: return f"{sec//60}m {sec%60:02d}s"
    return f"{sec//3600}h {(sec%3600)//60:02d}m"

class OMBox:
    def __init__(self):
        self.status, self.foot = "WORKING", "preparing"
        self.file = self.folder = ""
        self.done = self.total = 0
        self.speed = 0.0
        self.eta = None
        self.hist = []
        self.queue_left = 0
        self._h = None
        self.paint()
    def paint(self):
        pct = 0 if not self.total else min(100.0, 100.0 * self.done / self.total)
        hist = "".join(f'<div class="om-hrow"><span class="om-hk">{esc(a)}</span><span class="om-hv">{esc(b)}</span></div>' for a, b in self.hist[-12:])
        extra = ""
        if self.file:
            extra = f"""
            <div class="om-name">{esc(self.file)}</div>
            <div class="om-sub">{esc(self.folder)}{('  ·  ' + str(self.queue_left) + ' queued') if self.queue_left else ''}</div>
            <div class="om-bar-track"><div class="om-bar-fill" style="width:{pct:.2f}%"></div></div>
            <div class="om-stat">{fmt_bytes(self.done)} / {fmt_bytes(self.total) if self.total else '—'}  ·  {fmt_bytes(self.speed)}/s  ·  {fmt_eta(self.eta)} remaining  ·  {pct:.1f}%</div>
            """
        if self.hist:
            extra += f'<div class="om-hist">{hist}</div>'
        if not self.file and not self.hist:
            extra = '<div class="om-note">Preparing downloads.</div>'
        h = HTML(CSS + f"""
        <div class="om-wrap">
          <div class="om-top"><div class="om-kicker">02</div>
            <div class="om-status om-s-{esc(self.status)}">{esc(self.status)}</div></div>
          <div class="om-title">Models</div>
          <div class="om-rule"></div>
          {extra}
          <div class="om-foot">{esc(self.foot)}</div>
        </div>""")
        self._h = display(h, display_id="sn-02") if self._h is None else self._h.update(h) or self._h

ui = OMBox()

if include_patches:
    try:
        os.makedirs(PATCHES, exist_ok=True)
        tree = requests.get(f"https://huggingface.co/api/models/{HF_REPO}/tree/main?recursive=true", timeout=60).json()
        plist = sorted((f for f in tree if isinstance(f, dict) and f.get("type") == "file"
                        and str(f.get("path", "")).startswith("patches/") and f["path"].endswith(".patch")),
                       key=lambda f: f["path"])
        got = 0
        for f in plist:
            dest = os.path.join(PATCHES, os.path.basename(f["path"]))
            if not (os.path.isfile(dest) and (not f.get("size") or os.path.getsize(dest) == f["size"])):
                r = requests.get(HF_BASE + f["path"], timeout=120)
                r.raise_for_status()
                with open(dest, "wb") as fh:
                    fh.write(r.content)
            got += 1
        note = f"{got} .patch  ·  " + ("applied in 01" if os.path.isfile(MARK) else "on disk, applies in 01")
        ui.hist.append(("patches", note))
    except Exception as e:
        ui.hist.append(("patches", f"failed  ·  {e}"))
    ui.paint()

def resolve_job(url, dest_dir):
    r = requests.head(url, allow_redirects=True, timeout=60)
    final, size = r.url, int(r.headers.get("Content-Length") or 0)
    if r.status_code >= 400 or not size:
        r = requests.get(url, stream=True, timeout=60)
        final, size = r.url, int(r.headers.get("Content-Length") or 0)
        r.close()
    name = os.path.basename(unquote(urlparse(url).path)) or "download.bin"
    return {"final": final, "name": name, "size": size,
            "dest": os.path.join(dest_dir, name), "dir": dest_dir,
            "headers": {"User-Agent": "Mozilla/5.0"}}

def download_stream(job, progress):
    dest, url, hdrs = job["dest"], job["final"], job["headers"]
    tmp = dest + ".part"
    with requests.get(url, headers=hdrs, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length") or job["size"] or 0)
        done = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1024 * 256):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                progress(done, total)
    os.replace(tmp, dest)
    progress(os.path.getsize(dest), os.path.getsize(dest))

def download_parallel(job, progress, n=8):
    dest, url, hdrs, size = job["dest"], job["final"], job["headers"], job["size"]
    if not size or size < 8 * 1024 * 1024:
        return download_stream(job, progress)
    probe = requests.head(url, headers=hdrs, allow_redirects=True, timeout=30)
    if probe.headers.get("Accept-Ranges", "").lower() != "bytes":
        return download_stream(job, progress)
    tmp = dest + ".part"
    with open(tmp, "wb") as f:
        f.truncate(size)
    lock = threading.Lock()
    done = [0]
    err = [None]
    chunk = (size + n - 1) // n

    def worker(i):
        start = i * chunk
        end = min(start + chunk, size) - 1
        if start > end:
            return
        h = dict(hdrs)
        h["Range"] = f"bytes={start}-{end}"
        for attempt in range(4):
            try:
                with requests.get(url, headers=h, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    with open(tmp, "r+b") as f:
                        f.seek(start)
                        for part in r.iter_content(1024 * 256):
                            if not part:
                                continue
                            f.write(part)
                            with lock:
                                done[0] += len(part)
                                progress(done[0], size)
                return
            except Exception as e:
                err[0] = e
                time.sleep(1.5 * (attempt + 1))
        raise err[0]

    with ThreadPoolExecutor(max_workers=n) as ex:
        futs = [ex.submit(worker, i) for i in range(n)]
        for fut in futs:
            fut.result()
    os.replace(tmp, dest)

jobs = [(HF_BASE + f"SenseNova-Loop-9B-abliterated-{quant}.gguf", MODELS)]
if DRAFTERS.get(drafter):
    jobs.append((HF_BASE + DRAFTERS[drafter], MODELS))

state = {"done": 0, "total": 0, "t0": time.time(), "ema": 0.0, "last": 0, "lt": time.time()}
lock = threading.Lock()

def on_progress(done, total):
    now = time.time()
    with lock:
        state["done"] = done
        state["total"] = total or 0
        dt = now - state["lt"]
        if dt >= 0.15:
            inst = (done - state["last"]) / dt if dt > 0 else 0
            state["ema"] = inst if state["ema"] == 0 else (0.25 * inst + 0.75 * state["ema"])
            state["last"] = done
            state["lt"] = now

t0 = time.time()
failed = 0
for i, (url, folder) in enumerate(jobs):
    os.makedirs(folder, exist_ok=True)
    ui.queue_left = len(jobs) - i - 1
    ui.folder = folder
    ui.file = "resolving"
    ui.done = ui.total = 0
    ui.speed = 0
    ui.eta = None
    ui.foot = f"file {i+1} of {len(jobs)}"
    ui.paint()
    try:
        job = resolve_job(url, folder)
    except Exception as e:
        ui.hist.append((url, f"resolve failed  ·  {e}"))
        failed += 1
        ui.paint()
        continue

    ui.file = job["name"]
    ui.total = job["size"]
    ui.paint()

    if skip_existing and os.path.isfile(job["dest"]):
        existing = os.path.getsize(job["dest"])
        if not job["size"] or existing == job["size"]:
            ui.hist.append((job["name"], "already present"))
            ui.paint()
            continue

    state.update({"done": 0, "total": job["size"], "t0": time.time(), "ema": 0.0, "last": 0, "lt": time.time()})
    err_box = [None]

    def work():
        try:
            download_parallel(job, on_progress)
        except Exception as e:
            err_box[0] = e

    th = threading.Thread(target=work, daemon=True)
    th.start()
    while th.is_alive():
        with lock:
            ui.done, ui.total = state["done"], state["total"]
            ui.speed = state["ema"]
            ui.eta = (ui.total - ui.done) / ui.speed if ui.speed > 1 else None
        ui.paint()
        time.sleep(0.12)
    th.join()
    if err_box[0]:
        if os.path.isfile(job["dest"] + ".part"):
            try: os.remove(job["dest"] + ".part")
            except OSError: pass
        ui.hist.append((job["name"], f"failed  ·  {err_box[0]}"))
        failed += 1
    else:
        ui.hist.append((job["name"], f"{fmt_bytes(os.path.getsize(job['dest']))}  →  models"))
    ui.paint()

ui.file = ""
ui.status = "ERROR" if failed else "READY"
ui.foot = f"{len(jobs) - failed} of {len(jobs)} placed  ·  {int(time.time()-t0)}s  ·  next: cell 03"
ui.paint()


In [ ]:
#@title 03  ·  Launch { display-mode: "form" }
port = 8080 #@param {type:"integer"}
context = 8192 #@param {type:"integer"}
gpu_layers = 99 #@param {type:"integer"}
quant = "Q4_K_M" #@param ["Q4_K_M", "Q6_K"]
drafter = "drafter-2" #@param ["drafter-2", "drafter", "none"]
flash_attn = True #@param {type:"boolean"}
safe_mode = True #@param {type:"boolean"}
extra_args = "" #@param {type:"string"}

import os, re, sys, json, time, html, shlex, signal, socket, threading, subprocess, warnings
warnings.filterwarnings("ignore")

from collections import deque
from IPython.display import display, HTML, Javascript

ROOT = "/content/SenseNovaLoop"
MODELS = os.path.join(ROOT, "models")
BIN = os.path.join(ROOT, "llama.cpp-loop", "build", "bin", "llama-server")
CF = "/usr/local/bin/cloudflared"
PORT = int(port or 8080)
CTX = int(context or 8192)
NGL = int(gpu_layers if gpu_layers is not None else 99)
DRAFTERS = {
    "drafter-2": "SenseNova-Loop-9B-abliterated-drafter-2-Q4_K_M.gguf",
    "drafter": "SenseNova-Loop-9B-abliterated-drafter-Q4_K_M.gguf",
    "none": "",
}
DRAFT = DRAFTERS.get(drafter, "")
MODEL = os.path.join(MODELS, f"SenseNova-Loop-9B-abliterated-{quant}.gguf")
DRAFT_PATH = os.path.join(MODELS, DRAFT) if DRAFT else ""
LOGS = deque(maxlen=400)
LOG_SEQ = [0]
PROCS = []
LOG_PATH = os.path.join(ROOT, "server.log")
def log_add(msg):
    LOGS.append(msg)
    LOG_SEQ[0] += 1
    try:
        with open(LOG_PATH, "a") as f:
            f.write(msg + "\n")
    except Exception:
        pass
MID = f"sn03_{int(time.time() * 1000)}"

CSS = f"""
<style>
#{MID} {{
  width:98%;margin:0 auto;box-sizing:border-box;background:#141210;color:#e8dfd0;
  border:1px solid #3f382c;padding:28px 32px 22px;
  font-family:Palatino,"Palatino Linotype","Book Antiqua",Georgia,serif;
}}
#{MID} .om-top{{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}}
#{MID} .om-kicker{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.28em;color:#b8956c;text-transform:uppercase;}}
#{MID} .om-status{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;}}
#{MID} .om-s-WORKING{{color:#b8956c;}} #{MID} .om-s-READY,#{MID} .om-s-LIVE{{color:#8fa382;}}
#{MID} .om-s-ERROR{{color:#8a3a3a;}} #{MID} .om-s-STOPPED{{color:#8a8070;}}
#{MID} .om-title{{font-size:28px;font-weight:500;letter-spacing:.04em;color:#f3eadb;margin:6px 0 0;line-height:1.15;}}
#{MID} .om-rule{{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}}
#{MID} a.om-url{{display:none;font-size:20px;color:#f3eadb;word-break:break-all;line-height:1.35;
  text-decoration:none;border-bottom:1px solid #5a4e3c;cursor:pointer;user-select:text;width:fit-content;max-width:100%;}}
#{MID} a.om-url:hover{{color:#fff8ee;border-bottom-color:#b8956c;}}
#{MID} .om-wait{{color:#a89880;font-size:15px;line-height:1.5;}}
#{MID} .om-sub{{margin-top:8px;color:#8a8070;font-size:15px;}}
#{MID} .om-loglab{{font-family:"Courier New",Courier,monospace;font-size:10px;letter-spacing:.22em;text-transform:uppercase;color:#b8956c;margin:18px 0 8px;}}
#{MID} .om-log{{background:#0e0c0a;border:1px solid #2a261f;padding:12px 14px;height:340px;max-height:340px;overflow-y:auto;overflow-x:hidden;
  font-family:"Courier New",Courier,monospace;font-size:11px;line-height:1.55;color:#c8bba6;white-space:pre-wrap;word-break:break-word;
  overscroll-behavior:contain;overflow-anchor:none;}}
#{MID} .om-foot{{margin-top:16px;font-family:"Courier New",Courier,monospace;font-size:11px;color:#8a8070;letter-spacing:.06em;text-transform:uppercase;}}
</style>
"""

class OMBox:
    def __init__(self):
        self.status, self.url, self.sub, self.foot = "WORKING", "", "starting", ""
        self._seq = 0
        self._sig = None
        display(HTML(CSS + f"""
        <div id="{MID}">
          <div class="om-top"><div class="om-kicker">03</div>
            <div class="om-status om-s-WORKING" data-k="status">WORKING</div></div>
          <div class="om-title">Server</div>
          <div class="om-rule"></div>
          <a class="om-url" data-k="url" href="about:blank" target="_blank" rel="noopener noreferrer" referrerpolicy="no-referrer"></a>
          <div class="om-wait" data-k="wait">waiting for a public url</div>
          <div class="om-sub" data-k="sub">starting</div>
          <div class="om-loglab">Log</div>
          <div class="om-log" data-k="log"></div>
          <div class="om-foot" data-k="foot"></div>
        </div>
        <script>
        (function(){{
          var root = document.getElementById("{MID}");
          if (!root) return;
          var log = root.querySelector('[data-k=log]');
          var follow = true;
          log.addEventListener('scroll', function(){{
            follow = (log.scrollHeight - log.scrollTop - log.clientHeight) < 48;
          }});
          var a = root.querySelector('[data-k=url]');
          a.addEventListener('click', function(ev){{
            var href = a.getAttribute('href');
            if (!href || href === 'about:blank') return;
            ev.preventDefault();
            window.open(href, '_blank', 'noopener,noreferrer');
          }});
          window.__sn03Apply_{MID} = function(d){{
            var st = root.querySelector('[data-k=status]');
            st.textContent = d.status;
            st.className = 'om-status om-s-' + d.status;
            var w = root.querySelector('[data-k=wait]');
            if (d.url) {{
              a.href = d.url;
              a.textContent = d.url;
              a.style.display = 'inline';
              w.style.display = 'none';
            }} else {{
              a.style.display = 'none';
              a.removeAttribute('href');
              a.textContent = '';
              w.style.display = 'block';
            }}
            root.querySelector('[data-k=sub]').textContent = d.sub || '';
            root.querySelector('[data-k=foot]').textContent = d.foot || '';
            if (log.textContent !== d.log) {{
              log.textContent = d.log;
              if (follow) requestAnimationFrame(function(){{ log.scrollTop = log.scrollHeight; }});
            }}
          }};
        }})();
        </script>
        """), display_id=MID)
        display(Javascript("void 0"), display_id=MID + "_js")
        self.paint(force=True)

    def paint(self, force=False):
        payload = {
            "status": self.status,
            "url": self.url,
            "sub": self.sub,
            "foot": self.foot,
            "log": "\n".join(LOGS),
        }
        sig = (payload["status"], payload["url"], payload["sub"], payload["foot"], payload["log"])
        if not force and sig == self._sig:
            return
        self._sig = sig
        self._seq += 1
        code = f"window.__sn03Apply_{MID} && window.__sn03Apply_{MID}({json.dumps(payload)}); /*{self._seq}*/"
        display(Javascript(code), display_id=MID + "_js", update=True)

ui = OMBox()

def pump(proc, tag):
    try:
        for line in iter(proc.stdout.readline, ""):
            if not line:
                break
            log_add(f"{tag}  {line.rstrip()}")
    except Exception:
        pass

def bindable(p):
    s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        s.bind(("127.0.0.1", p)); return True
    except Exception:
        return False
    finally:
        s.close()

def pick_port(p):
    for cand in range(p, p + 24):
        if bindable(cand):
            return cand
    return p

def kill_tree(proc):
    if not proc or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        try: proc.terminate()
        except Exception: pass
    try:
        proc.wait(timeout=4)
    except Exception:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            try: proc.kill()
            except Exception: pass

def cleanup():
    for p in list(PROCS):
        kill_tree(p)
    PROCS.clear()
    subprocess.run(f"pkill -f {shlex.quote(CF)}", shell=True, capture_output=True)
    subprocess.run(f"pkill -f {shlex.quote(BIN)}", shell=True, capture_output=True)

def port_open(p):
    s = socket.socket(); s.settimeout(1.0)
    try:
        s.connect(("127.0.0.1", p)); s.close(); return True
    except Exception:
        return False

def http_ready(p):
    import urllib.request
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{p}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def gpu_mem_mb():
    try:
        o = subprocess.run(
            "nvidia-smi --query-gpu=memory.total,memory.free --format=csv,noheader,nounits",
            shell=True, capture_output=True, text=True,
        ).stdout.strip()
        total, free = [int(x) for x in o.splitlines()[0].split(",")]
        return total, free
    except Exception:
        return 0, 0

def ram_free_mb():
    try:
        vals = {}
        for line in open("/proc/meminfo"):
            k, v = line.split(":", 1)
            vals[k] = int(v.strip().split()[0]) // 1024
        return vals.get("MemAvailable", 0)
    except Exception:
        return 0

def ensure_cloudflared():
    if os.path.isfile(CF) and os.access(CF, os.X_OK):
        return
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        shell=True,
    )
    subprocess.run("chmod +x /usr/local/bin/cloudflared", shell=True)

def find_url():
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", "\n".join(LOGS))
    return m.group(0) if m else None

def tunnel_ready():
    return bool(re.search(
        r"Connection registered|Registered tunnel connection|Registered conn|connIndex\s*=\s*\d+",
        "\n".join(LOGS), re.I,
    ))

def tunnel_auth_fail():
    return bool(re.search(
        r"unauthorized|no authorization|login required|failed to request a quick|couldn't request a tunnel|429|Too many",
        "\n".join(LOGS), re.I,
    ))

def wait_on_logs(seconds, pred):
    deadline = time.time() + seconds
    last = -1
    while time.time() < deadline:
        if pred():
            return True
        if LOG_SEQ[0] != last:
            last = LOG_SEQ[0]
            ui.paint()
        time.sleep(0.35)
    return False

def start_cloudflared(extra):
    args = [CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"] + extra
    log_add("tunnel  " + " ".join(args[1:]))
    proc = subprocess.Popen(
        args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    PROCS.append(proc)
    threading.Thread(target=pump, args=(proc, "tunnel"), daemon=True).start()
    return proc

def open_cloudflare():
    attempts = [
        ["--protocol", "http2"],
        ["--protocol", "http2", "--edge-ip-version", "4"],
        [],
    ]
    for i, extra in enumerate(attempts, 1):
        log_add(f"cloudflare  attempt {i}/{len(attempts)}")
        ui.paint()
        proc = start_cloudflared(extra)
        wait_on_logs(80, lambda: (find_url() and tunnel_ready()) or tunnel_auth_fail() or proc.poll() is not None)
        url = find_url()
        if url and (tunnel_ready() or wait_on_logs(20, tunnel_ready)):
            log_add("cloudflare  edge connection registered")
            return proc, url
        if url and proc.poll() is None and not tunnel_auth_fail():
            log_add("cloudflare  url printed  ·  waiting briefly for edge")
            time.sleep(4)
            if proc.poll() is None:
                return proc, url
        reason = "authorization failed" if tunnel_auth_fail() else "no registered connection"
        log_add(f"cloudflare  attempt {i} failed  ·  {reason}")
        kill_tree(proc)
        if proc in PROCS:
            PROCS.remove(proc)
        time.sleep(2.0)
    return None, None

def fail(msg):
    ui.status, ui.sub = "ERROR", msg
    ui.paint(force=True)
    cleanup()

def build_cmd():
    cmd = [BIN, "-m", MODEL, "-c", str(CTX), "-ngl", str(NGL),
           "--host", "127.0.0.1", "--port", str(PORT),
           "--jinja", "-fa", "on" if flash_attn else "off",
           "--parallel", "1", "--cache-ram", "0",
           "--temp", "0.6", "--top-p", "0.95", "--top-k", "20", "--min-p", "0"]
    if not safe_mode:
        cmd += ["--ctx-checkpoints", "3", "--ctx-checkpoint-turns", "--prompt-tile", "32"]
        if DRAFT_PATH:
            cmd += ["-md", DRAFT_PATH, "--spec-type", "ngram-simple,draft-dflash",
                    "--spec-ngram-simple-size-m", "31", "--spec-draft-n-max", "7",
                    "--spec-draft-n-ubatch", "64"]
        else:
            cmd += ["--spec-type", "ngram-simple", "--spec-ngram-simple-size-m", "31"]
        cmd += ["--spec-rollback", "31"]
    if (extra_args or "").strip():
        cmd += shlex.split(extra_args)
    return cmd

def startup_fail(msg):
    log_add("guard  " + msg)
    ui.status, ui.sub, ui.foot = "ERROR", msg, "nothing was started"
    ui.paint(force=True)
    raise SystemExit

def _main():
    global PORT
    if not os.path.isfile(BIN):
        startup_fail("llama-server is not built  ·  run cell 01")
    if not os.path.isfile(MODEL):
        startup_fail(f"{os.path.basename(MODEL)} missing  ·  run cell 02")
    if DRAFT_PATH and not os.path.isfile(DRAFT_PATH):
        startup_fail(f"{DRAFT} missing  ·  run cell 02 with this drafter selected")

    try:
        open(LOG_PATH, "w").close()
    except Exception:
        pass

    ensure_cloudflared()
    cleanup()
    time.sleep(0.4)
    PORT = pick_port(PORT)
    vram_total, vram_free = gpu_mem_mb()
    ram = ram_free_mb()
    log_add(f"port  {PORT}  ·  ram {ram} MB available" + (f"  ·  vram {vram_free}/{vram_total} MB free" if vram_total else "  ·  no gpu"))
    ui.foot = "interrupt this cell to stop the server and the tunnel"
    ui.paint(force=True)
    if vram_total and vram_free < 5500:
        fail("not enough free vram for the weights")
        raise SystemExit
    if ram and ram < 2200:
        fail("system ram too low")
        raise SystemExit

    cmd = build_cmd()
    log_add("exec  " + " ".join(cmd))
    label = "safe mode  ·  no drafts" if safe_mode else (f"{quant} + {drafter}" if DRAFT_PATH else f"{quant}  ·  ngram drafts only")
    try:
        srv = subprocess.Popen(
            cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, start_new_session=True, env=os.environ.copy(),
        )
        PROCS.append(srv)
        threading.Thread(target=pump, args=(srv, "server"), daemon=True).start()

        ui.sub = f"{label}  ·  ctx {CTX}  ·  loading weights"
        ui.paint(force=True)

        wait_on_logs(600, lambda: srv.poll() is not None or (port_open(PORT) and http_ready(PORT)))
        if srv.poll() is not None:
            fail("llama-server exited before it became ready")
        elif not http_ready(PORT):
            fail("timed out waiting for the model to load")
        else:
            _, vram_free = gpu_mem_mb()
            log_add(f"model loaded  ·  server is accepting connections  ·  vram {vram_free} MB free")
            ui.sub = "server up  ·  opening cloudflare tunnel"
            ui.paint(force=True)

            tunnel, public = open_cloudflare()
            if not public:
                fail("cloudflare did not publish a url")
            else:
                ui.status = "LIVE"
                ui.url = public
                ui.sub = f"{label}  ·  ctx {CTX}  ·  cloudflare up  ·  click the url to open"
                ui.foot = "interrupt this cell to stop the server and the tunnel"
                ui.paint(force=True)
                last = -1
                while True:
                    if srv.poll() is not None:
                        ui.status, ui.sub = "ERROR", "llama-server process ended"
                        ui.paint(force=True)
                        break
                    if tunnel.poll() is not None:
                        ui.status, ui.sub = "ERROR", "tunnel process ended"
                        ui.paint(force=True)
                        break
                    if LOG_SEQ[0] != last:
                        last = LOG_SEQ[0]
                        ui.paint()
                    time.sleep(0.4)
    except KeyboardInterrupt:
        log_add("interrupt received")
    except Exception as e:
        log_add(f"error  {e}")
        ui.status, ui.sub = "ERROR", "unexpected error"
    finally:
        cleanup()
        if ui.status != "ERROR":
            ui.status = "STOPPED"
            ui.url = ""
            ui.sub = f"server and tunnel stopped  ·  port {PORT} released"
            ui.foot = "run this cell again to start"
        ui.paint(force=True)

try:
    _main()
except SystemExit:
    pass